In [1]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.fkhmqgc2M3/ipykernel_3195734/1270949041.py:4: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration

# Read in dataset

In [3]:
d = gpd.read_file("/home/csutter/DRIVE-clean/NWS_warnings/data/nws_warnings/nws_all_warnings_cleaned.gpkg")

d.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d["duration"] = pd.to_timedelta(d["duration_sec"], unit="s")

# may take ~40 seconds

Grab some basic counts for reference

In [14]:
# Explore what warnings there are (no need to run)

print(np.unique(d["name"]))
# Yes: 'Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory'
# Maybe: 'Cold Weather Advisory', 'Ice Storm Warning'

eventsofint = ['Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory']

devents = d[d["name"].isin(eventsofint)]

print(len(devents)) # note that this isn't the number of instances to run for inference. Because 1) events elements are a range of time, so need to run multiple 5-min instances within one 'event', so this makes *more* instances, 2) If there an event that spans multiple regions, there is likely overlapping times of the warning. These will end up being duplicate because when we prepare the inference instances to run (below) we run the entire state for every time where there was an active snow squall warning, regardless of where the warning was (cleaner to maintain consistency in inference runs by running every instance statewide)

display(devents.groupby(["name"]).count())

['Beach Hazard Statement' 'Blizzard Warning' 'Coastal Flood Advisory'
 'Coastal Flood Statement' 'Coastal Flood Warning' 'Coastal Flood Watch'
 'Cold Weather Advisory' 'Dense Fog Advisory' 'Excessive Heat Warning'
 'Extreme Heat Warning' 'Extreme Heat Watch' 'Fire Weather Watch'
 'Flash Flood Warning' 'Flood Advisory' 'Flood Warning' 'Flood Watch'
 'Freeze Warning' 'Freeze Watch' 'Frost Advisory' 'Heat Advisory'
 'High Surf Advisory' 'High Wind Warning' 'High Wind Watch'
 'Ice Storm Warning' 'Lake Effect Snow Warning' 'Lakeshore Flood Advisory'
 'Lakeshore Flood Warning' 'Lakeshore Flood Watch' 'Red Flag Warning'
 'Rip Currents Statement' 'Severe Thunderstorm Warning'
 'Severe Thunderstorm Watch' 'Snow Squall Warning' 'Tornado Warning'
 'Tornado Watch' 'Wind Advisory' 'Wind Chill Advisory'
 'Wind Chill Warning' 'Wind Chill Watch' 'Winter Storm Warning'
 'Winter Storm Watch' 'Winter Weather Advisory']
6548


,Unnamed: 0,vtec_year,iso_issued,issued,iso_expired,expired,eventid,phenomena,significance,hvtec_nwsli,...,product_id,ph_name,sig_name,url,location_type,ugc_gis,loc_desc,duration_sec,geometry,duration
name,,,,,,,,,,,,,,,,,,,,,
Blizzard Warning,13,13,13,13,13,13,13,13,13,0,...,13,13,13,13,13,13,13,13,13,13
Dense Fog Advisory,639,639,639,639,639,639,639,639,639,0,...,639,639,639,639,639,639,639,639,639,639
Lake Effect Snow Warning,219,219,219,219,219,219,219,219,219,0,...,219,219,219,219,219,219,219,219,219,219
Snow Squall Warning,716,716,716,716,716,716,716,716,716,0,...,716,716,716,716,716,716,716,716,716,716
Winter Storm Warning,886,886,886,886,886,886,886,886,886,0,...,886,886,886,886,886,886,886,886,886,886
Winter Storm Watch,1013,1013,1013,1013,1013,1013,1013,1013,1013,0,...,1013,1013,1013,1013,1013,1013,1013,1013,1013,1013
Winter Weather Advisory,3062,3062,3062,3062,3062,3062,3062,3062,3062,0,...,3062,3062,3062,3062,3062,3062,3062,3062,3062,3062


In [3]:
# some basic info, no need to run

squall = d[d["name"]=="Snow Squall Warning"]
print("number of snow squall entries (note tha these aren't unique events, b/c one event may have multiple locations/datetimes associated with it. E.g. if squall Saratoga warning was issue at 10am, and Albany warning was at 1015am. Or even if Albany was also at 10am, same time as the Saratoga issued time, it would still be two entries.")
print(len(squall))
print("unique locations since 2022 that have had snow squalls")
print(len(np.unique(squall["loc_desc"])))
print("unique datetimes since 2022 that have had snow squalls")
print(len(np.unique(squall["issued"])))

blizzard = d[d["name"]=="Blizzard Warning"]
print(len(blizzard))

ws_warn = d[d["name"]=="Winter Storm Warning"]
print(len(ws_warn))

ws_watch = d[d["name"]=="Winter Storm Watch"]
print(len(ws_watch))

ww_advisory = d[d["name"]=="Winter Weather Advisory"]
print(len(ww_advisory))

densefog = d[d["name"]=="Dense Fog Advisory"]
print(len(densefog))

group

number of snow squall entries
716
unique locations since 2022 that have had snow squalls
61
unique datetimes since 2022 that have had snow squalls
200
13
886
1013
3062
639


# Grab events to run for inference 
- Using dataframe read in above
- Overview of flow of prepping data: Must subset to events you want to run/organize in one "launch" of inference jobs. Often, I want to run 3 jobs simulatenously on Hulk. This code below prepares for that level of run. Why does that matter? Because when removing the datetimes that have already had inference runs on, that step uses the "aggregated" dir (/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns") which already has consolidate across all the different inference sets, thus the aggregation code needs to have been ran up to that point, so the jobs we're preparing to run here exclude ONLY those that are already DONE running and in aggregation (i.e. you can't start one job for lake effect events, and then prep a new dataset for blizzard events and run that one 3 mins later, bc there may be overlap between the lake effect and blizzard events, and you'd then be running duplicate code.)
- TL;DR - Just make sure you run the aggregated dir code before running this code, and make sure that a job you want to start from this code doesn't rely on/ have duplicate datetimes with another job that is already running. In other words, The d_event_subset below needs to contain the full set from which you will start all your jobs right now.


Events for tracking
- 'Snow Squall Warning' (RAN Dec 2025)
- 'Blizzard Warning' (small set) (RAN Jan 2026)
- 'Lake Effect Snow Warning'
- 'Winter Storm Warning' 
- STILL TO RUN AS OF 1/21/26: 'Winter Storm Watch', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory'

In [4]:
# grab snow squall data (or whatever event of interest is)
d_event_subset = d[d["name"]=='Winter Storm Warning'] # ADJUST HERE!!

# just for reference
print(len(d_event_subset))

886


In [5]:
# Grab start and end times, for which we'll run all datetimes in between
# Will grab start time, end time, and list every 5-min increment in between
# Will also run 15 mins before and after squall starts to capture the prior and post conditions
# Round to 0005, 0010, etc, 5 min increments. Why? B/c 1) we only snapshot images in 5-min increments anyway, and while they won't be exactly on the even 5 mins, if we consistently run inference runs with ever 5 mins, it means we'll capture all instances (e.g. if we started allowing "off" times like 11 min rather than 10, that may be the same "image instance" for 1000 cams -- this is a waste of inference run). This allows us to keep track seamlessly across lots of different case study inference runs, exactly which datetimes have and have not been ran already
# Note: we run every instance statewide, just for consistency/ease of run (see note above about seamless tracking across case studies). I'm not parsing certain regions based on those that had the squall in region. 

d_event_subset["start_round"] = d_event_subset["issued"].dt.round("5min") # will want these in a df for reference of the rounded start time
d_event_subset["end_round"] = d_event_subset["expired"].dt.round("5min") # will want these in a df for reference of the rounded ende time

d_event_subset["start_buffer"] = d_event_subset["start_round"] - pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] + pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="5T"),
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

,Unnamed: 0,vtec_year,iso_issued,issued,iso_expired,expired,eventid,phenomena,significance,hvtec_nwsli,...,loc_desc,duration_sec,geometry,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
348,7,2022,2022-02-03T22:00:00Z,2022-02-03 22:00:00,2022-02-05T00:38:00Z,2022-02-05 00:38:00,3,WS,W,None,...,Eastern Rensselaer,95880.0,"POLYGON ((-73.27460 42.94311, -73.27450 42.939...",1 days 02:38:00,2022-02-03 22:00:00,2022-02-05 00:40:00,2022-02-03 21:45:00,2022-02-05 00:55:00,"DatetimeIndex(['2022-02-03 21:45:00', '2022-02...","[20220203_2145, 20220203_2150, 20220203_2155, ..."
355,14,2022,2022-02-25T04:00:00Z,2022-02-25 04:00:00,2022-02-25T20:28:00Z,2022-02-25 20:28:00,4,WS,W,None,...,Eastern Rensselaer,59280.0,"POLYGON ((-73.27460 42.94311, -73.27450 42.939...",0 days 16:28:00,2022-02-25 04:00:00,2022-02-25 20:30:00,2022-02-25 03:45:00,2022-02-25 20:45:00,"DatetimeIndex(['2022-02-25 03:45:00', '2022-02...","[20220225_0345, 20220225_0350, 20220225_0355, ..."
359,18,2022,2022-03-12T09:00:00Z,2022-03-12 09:00:00,2022-03-13T03:17:00Z,2022-03-13 03:17:00,5,WS,W,None,...,Eastern Rensselaer,65820.0,"POLYGON ((-73.27460 42.94311, -73.27450 42.939...",0 days 18:17:00,2022-03-12 09:00:00,2022-03-13 03:15:00,2022-03-12 08:45:00,2022-03-13 03:30:00,"DatetimeIndex(['2022-03-12 08:45:00', '2022-03...","[20220312_0845, 20220312_0850, 20220312_0855, ..."
380,39,2022,2022-12-16T00:00:00Z,2022-12-16 00:00:00,2022-12-17T11:31:00Z,2022-12-17 11:31:00,7,WS,W,None,...,Eastern Rensselaer,127860.0,"POLYGON ((-73.27460 42.94311, -73.27450 42.939...",1 days 11:31:00,2022-12-16 00:00:00,2022-12-17 11:30:00,2022-12-15 23:45:00,2022-12-17 11:45:00,"DatetimeIndex(['2022-12-15 23:45:00', '2022-12...","[20221215_2345, 20221215_2350, 20221215_2355, ..."


In [6]:
# make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

datetimes_all = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        datetimes_all.append(j)

In [7]:

dates_list = np.unique(datetimes_all)
print(len(dates_list)) # this is the amount of instances to run!

print(len(datetimes_all))
# Note that some will be overlapping for nearby counties or overlapping zones

print(dates_list[0:4])


# Need to also see which datetimes I've already ran from past inference! See other notebook. 


17108
252662
['20220101_2345' '20220101_2350' '20220101_2355' '20220102_0000']


In [33]:
# just for investigation, dont need to run

filepath = glob(f"/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/*/*/*/*") # just need the datetime which is in the path name, not all the way down the csv pred level

fs= sorted(filepath)

fs[0:4]

['/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/12/20220112_0000',
 '/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/19/20220119_2000',
 '/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/20/20220120_1600',
 '/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/20/20220120_2000']

In [7]:
# remove instances for which we already ran inference on in past studies -- would be duplicate, not needed

# REFERENCE: from this code: /home/csutter/DRIVE-clean/operational_runs/dates_sample.ipynb

# Grab dates already ran
# Note that you will have needed to make sure all runs were aggregated into the agg dir, as that is where this code "looks" for dates already ran. That is done in this file: /home/csutter/DRIVE-clean/operational_analysis/notebooks/analysis.ipynb

from glob import glob

scm_finalpreds = []

filepath = glob(f"/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/*/*/*/*") # just need the datetime which is in the path name, not all the way down the csv pred level

# print(filepath)
# print(len(filepath))

for f in filepath:
    # print(f)
    f = f[-13:]
    scm_finalpreds.append(f)


print("instances for which we have predictions")
print(len(scm_finalpreds))
scm_finalpreds[0]

print(scm_finalpreds[0:3])

# Cross check from dates_list and remove any dates already ran from scm_finalpreds

dates_list_new = []
for d in dates_list:
    if d not in scm_finalpreds:
        dates_list_new.append(d)

print("Unique datetimes")
print(len(dates_list_new))



instances for which we have predictions
4155
['20221217_2000', '20221217_0800', '20221217_0000']
Unique datetimes
956


In [8]:
print(len(dates_list))
print(len(dates_list_new))  # difference shows removed instances that were already ran

1012
956


In [12]:
# Save out 
forcsv = pd.DataFrame(dates_list_new[600:], columns=['date']) # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set28_blizzard3" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir)
forcsv.to_csv(f"{savetodir}/dates.csv") 